# Kaggle Submission

This notebook makes our late submission. It trains the baseline from
`assignment1_eda_baseline.ipynb` (Ridge, alpha = 100000) on all training days and
sends the predictions through the competition's evaluation API, as explained on the
[Data page](https://www.kaggle.com/competitions/mitsui-commodity-prediction-challenge/data).

To run it on Kaggle, our `src` files must be added to the notebook as a Kaggle dataset.

In [ ]:
import glob
import os
import sys

labels_file = glob.glob("/kaggle/input/**/train_labels.csv", recursive=True)[0]
DATA_DIR = os.path.dirname(labels_file)
sys.path.append(DATA_DIR)

models_file = glob.glob("/kaggle/input/**/models.py", recursive=True)[0]
sys.path.append(os.path.dirname(models_file))

print("Competition data:", DATA_DIR)
print("Our src files:   ", os.path.dirname(models_file))

In [ ]:
import pandas as pd
from data import load_data, preprocess
from models import train_baseline, predict as model_predict

train, labels, target_pairs = load_data(DATA_DIR)
features = preprocess(train)
targets = labels.drop(columns="date_id")

scaler, models = train_baseline(features, targets, alpha=100000)

In [ ]:
history = train.copy()


def predict(test, label_lags_1, label_lags_2, label_lags_3, label_lags_4):
    """Called by the API once per day. Returns one row with 424 predictions."""
    global history
    today = test.to_pandas()[train.columns].astype(float)
    history = pd.concat([history, today], ignore_index=True)
    today_features = preprocess(history).tail(1)
    return model_predict(scaler, models, today_features).reset_index(drop=True)

In [ ]:
import kaggle_evaluation.mitsui_inference_server

server = kaggle_evaluation.mitsui_inference_server.MitsuiInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    server.serve()
else:
    server.run_local_gateway((DATA_DIR,))